# Liu2024 Source MAT S-JEPA Representation Linear Probe

Frozen S-JEPA representation extraction followed by simple sklearn classifiers.

# 1. Setup

In [1]:
import os

import re
import sys
import json
import math
import hashlib
import random
import builtins
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, Subset, DataLoader

from scipy.io import loadmat
from scipy import signal

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from skorch.callbacks import EarlyStopping, EpochScoring
from skorch.dataset import ValidSplit

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal

import mne

mne.set_log_level("WARNING")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().resolve().parent.parent
print(f"\nWorking directory: {WORKING_DIR}")


Runtime Environment:
  - Python: 3.11.15 (main, Apr  9 2026, 01:18:52) [Clang 21.0.0 (clang-2100.0.123.102)]
  - Platform: macOS-26.5-arm64-arm-64bit

Working directory: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA


# 2. Configuration

In [3]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


In [4]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "baseline_sjepa_prelocal",
    "config_note": "Clean MNE-style preprocessing pipeline builder + S-JEPA hyperparameter controls.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "baseline_window_mean",      # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",                     # none, constant, linear
    "eog_correction": "none",                   # none, linear_regression

    # Robust source-domain clipping / winsorization. Use cautiously.
    "artifact_clip_mode": "none",               # none, absolute, percentile
    "artifact_clip_abs_value": None,             # in source_unit, e.g. 150.0 when source_unit=microvolts
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing
    # ------------------------------------------------------------------
    "reference_mode": "average",                # average, none
    "reference_timing": "before_resample_filter",  # before_resample_filter, after_resample_before_filter, after_filter
    "resample": True,
    "resample_sfreq": 128,

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",                     # fir, iir
    "filter_phase": "zero",                     # zero, zero-double, minimum (FIR only)
    "filter_fir_design": "firwin",              # firwin, firwin2 (FIR only)
    "filter_l_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_h_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_iir_params": None,                    # example: {"order": 2, "ftype": "butter"}

    "notch_freqs": None,                          # example: [50.0]
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing and post-window cleaning
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": 537,
    "mi_window_start_s": 2,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,       # in final_model_unit
    "reject_abs_threshold": None,                # in final_model_unit
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization. train_* modes are fit on each training split only.
    # ------------------------------------------------------------------
    "normalization_mode": "none",               # none, train_global_zscore, train_channel_zscore, train_channel_robust, trial_global_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                          # new, full
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",      # stratified_kfold, liu2024_repeated_60_40, repeated_stratified_split
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters
    # ------------------------------------------------------------------
    "batch_size": 16,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0004,
    "optimizer_name": "adam",                   # adam, adamw
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",           # valid_loss, valid_balanced_accuracy
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 0.0,

    # Lightweight training-time augmentation.
    "augmentation_noise_fraction": 0.0,
    "augmentation_time_shift_samples": 0,
    "augmentation_channel_dropout_prob": 0.0,

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics / interpretation
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": False,
    "plot_individual_spatial_filters": False,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,
    "topomap_value_mode": "relative_zscore",    # raw, relative_zscore, relative_percent
    "topomap_cmap": "RdBu_r",
    "collapse_threshold": 0.9,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

CONFIG.update({
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-representation-linear-probe"),
    "experiment_name": "liu2024_sjepa_representation_linear_probe",
    "config_note": "Frozen S-JEPA representation extraction with simple sklearn classifiers.",
    "demean_mode": "none",
    "target_window_samples": 537,
    "target_window_s": 4.2,
    "mi_window_start_s": 1.5,
    "filter_method": "fir",
    "filter_iir_params": None,
    "batch_size": 4,
    "learning_rate": 0.0003,
    "gradient_clip_norm": 0,
    "checkpoint_metric": "valid_loss",

    # Representation extraction.
    "representation_layer": "final_layer_input",  # final_layer_input, model_output, or a module name
    "representation_pooling": "flatten",          # flatten, mean_tokens
    "representation_batch_size": 16,

    # Simple classifiers to run on frozen features.
    "simple_classifiers": ["logistic_regression", "linear_svm", "ridge", "lda"],
})

In [5]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ  # compatibility with existing cells/artifacts

if CONFIG.get("target_window_samples", None) is None:
    WINDOW_SAMPLES = int(round(float(CONFIG["target_window_s"]) * EFFECTIVE_SFREQ))
else:
    WINDOW_SAMPLES = int(CONFIG["target_window_samples"])

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

PREPROCESSING_KEYS = [
    "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]

TRAINING_KEYS = [
    "strategy", "batch_size", "learning_rate", "optimizer_name", "weight_decay",
    "val_split", "early_stopping_patience", "n_epochs",
    "augmentation_noise_fraction", "augmentation_time_shift_samples", "augmentation_channel_dropout_prob",
]

EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
TRAINING_CONFIG = summarize_selected_config(TRAINING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

def print_config_block(title, values):
    print(title)
    for key, value in values.items():
        print(f"  {key:34s}: {value}")

print("Effective Liu2024 Source MAT settings:")
print(f"  Experiment:                        {CONFIG.get('experiment_name')}")
print(f"  Note:                              {CONFIG.get('config_note')}")
print(f"  Channels:                          {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:                     {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:                      {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:                   {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start / samples:         {CONFIG['mi_window_start_s']} s / {WINDOW_SAMPLES}")
print(f"  Effective window duration:         {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Evaluation mode:                   {CONFIG.get('evaluation_mode')}")
print(f"  Fixed seed:                        base={CONFIG.get('seed')} | cv={CONFIG.get('cv_random_state')} | split={CONFIG.get('split_random_state')} | val={CONFIG.get('val_split_random_state')}")
print_config_block("\nPreprocessing config:", PREPROCESSING_CONFIG)
print_config_block("\nTraining config:", TRAINING_CONFIG)
print_config_block("\nEvaluation config:", EVALUATION_CONFIG)


Effective Liu2024 Source MAT settings:
  Experiment:                        liu2024_sjepa_representation_linear_probe
  Note:                              Frozen S-JEPA representation extraction with simple sklearn classifiers.
  Channels:                          29
  Channel names:                     ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']
  Source sfreq:                      500 Hz
  Effective sfreq:                   128.0 Hz
  MI window start / samples:         1.5 s / 537
  Effective window duration:         4.1953 s
  Evaluation mode:                   stratified_kfold
  Fixed seed:                        base=2026 | cv=2026 | split=2026 | val=2026

Preprocessing config:
  source_unit                       : microvolts
  final_model_unit                  : microvolts
  demean_mode                       : none
  baseline_window_s      

In [6]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


[2026-06-04 19:09:19] Run ID:     20260604_1909_76736b0d
[2026-06-04 19:09:19] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d
[2026-06-04 19:09:19] Config:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/config.json


In [7]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


[2026-06-04 19:09:19] Using device: mps
[2026-06-04 19:09:21] Seed initialized: 2026


# 3. Load and Prepare Data

In [8]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


In [9]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),  # type: ignore
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_values_to_mne_volts(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("source_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e-6
    if unit in ("mv", "millivolt", "millivolts"):
        return arr * 1e-3
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported source_unit={config.get('source_unit')}")

def mne_volts_to_model_unit(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("final_model_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e6
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported final_model_unit={config.get('final_model_unit')}")

def _none_like(value):
    return value is None or str(value).lower() in ("none", "off", "false", "")

def build_preprocessing_pipeline(config):
    """Return a readable pipeline plan.

    The pipeline is represented as a list of dictionaries rather than hidden global logic.
    Every row is logged and saved in the run metadata through the preprocessing step list.
    """
    pipeline = []

    # Fixed source structure.
    pipeline.append({
        "stage": "source",
        "name": "select_eeg_channels",
        "description": "select Liu EEG channels, drop CPz reference, EOG, and marker before model input",
        "enabled": True,
    })

    pipeline.append({
        "stage": "source",
        "name": "demean",
        "mode": config.get("demean_mode", "none"),
        "baseline_window_s": config.get("baseline_window_s"),
        "enabled": not _none_like(config.get("demean_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "detrend",
        "mode": config.get("detrend_mode", "none"),
        "enabled": not _none_like(config.get("detrend_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "eog_correction",
        "mode": config.get("eog_correction", "none"),
        "enabled": not _none_like(config.get("eog_correction", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "artifact_clipping",
        "mode": config.get("artifact_clip_mode", "none"),
        "abs_value": config.get("artifact_clip_abs_value"),
        "percentile": config.get("artifact_clip_percentile"),
        "enabled": not _none_like(config.get("artifact_clip_mode", "none")),
    })

    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()
    if reference_timing == "before_resample_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "before_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({"stage": "mne_raw", "name": "resample", "sfreq": config.get("resample_sfreq"), "enabled": bool(config.get("resample", True))})

    if reference_timing == "after_resample_before_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    pipeline.append({
        "stage": "mne_raw",
        "name": "bandpass_filter",
        "enabled": bool(config.get("filter_enabled", True)),
        "l_freq": config.get("filter_low"),
        "h_freq": config.get("filter_high"),
        "method": config.get("filter_method"),
        "phase": config.get("filter_phase"),
        "fir_design": config.get("filter_fir_design"),
        "iir_params": config.get("filter_iir_params"),
    })

    if reference_timing == "after_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if not bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "after_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({
        "stage": "window",
        "name": "crop_fixed_mi_window",
        "start_s": config.get("mi_window_start_s"),
        "target_window_samples": config.get("target_window_samples"),
        "enabled": True,
    })

    pipeline.append({
        "stage": "window",
        "name": "bad_trial_rejection",
        "enabled": bool(config.get("reject_bad_trials", False)),
        "peak_to_peak_threshold": config.get("reject_peak_to_peak_threshold"),
        "abs_threshold": config.get("reject_abs_threshold"),
    })

    pipeline.append({
        "stage": "split",
        "name": "fold_safe_normalization",
        "mode": config.get("normalization_mode", "none"),
        "enabled": not _none_like(config.get("normalization_mode", "none")),
    })

    return pipeline

def describe_pipeline(pipeline):
    lines = []
    for step in pipeline:
        status = "ON" if step.get("enabled", False) else "off"
        parts = [f"[{status}] {step.get('stage')}::{step.get('name')}"]
        for key, value in step.items():
            if key not in ("stage", "name", "description", "enabled") and value is not None:
                parts.append(f"{key}={value}")
        if step.get("description"):
            parts.append(f"- {step['description']}")
        lines.append(" | ".join(parts))
    return lines

PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline:")
for line in describe_pipeline(PREPROCESSING_PIPELINE):
    print("  - " + line)

def apply_source_demean(X_eeg, subject_id, config, steps):
    mode = str(config.get("demean_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source demean skipped")
        return X

    if mode == "trial_mean":
        steps.append("source demean: subtract trial/channel mean over time")
        return X - X.mean(axis=-1, keepdims=True)

    if mode == "baseline_window_mean":
        baseline = config.get("baseline_window_s", [0.0, 2.0])
        if baseline is None or len(baseline) != 2:
            raise ValueError("baseline_window_s must be [start_s, stop_s] for baseline_window_mean.")
        start_s, stop_s = float(baseline[0]), float(baseline[1])
        start = int(round(start_s * LIU_SOURCE_SFREQ))
        stop = int(round(stop_s * LIU_SOURCE_SFREQ))
        if start < 0 or stop <= start or stop > X.shape[-1]:
            raise ValueError(f"Subject {subject_id}: invalid baseline_window_s={baseline} for source length {X.shape[-1]}")
        steps.append(f"source demean: subtract baseline mean {baseline}s")
        return X - X[:, :, start:stop].mean(axis=-1, keepdims=True)

    raise ValueError(f"Unsupported demean_mode={config.get('demean_mode')}")

def apply_source_detrend(X_eeg, subject_id, config, steps):
    mode = str(config.get("detrend_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)
    if mode in ("none", "off", "false"):
        steps.append("source detrend skipped")
        return X
    if mode == "constant":
        steps.append("source detrend: scipy.signal.detrend(type='constant')")
        return signal.detrend(X, axis=-1, type="constant")
    if mode == "linear":
        steps.append("source detrend: scipy.signal.detrend(type='linear')")
        return signal.detrend(X, axis=-1, type="linear")
    raise ValueError(f"Unsupported detrend_mode={config.get('detrend_mode')}")

def apply_eog_correction(X_eeg, rawdata, subject_id, config, steps):
    mode = str(config.get("eog_correction", "none")).lower()
    if mode in ("none", "off", "false"):
        steps.append("EOG correction skipped")
        return X_eeg

    if mode != "linear_regression":
        raise ValueError(
            "Only eog_correction='linear_regression' is implemented in this source-MAT notebook. "
            "ICA is intentionally not included because the source pipeline drops EOG before model input and "
            "the dataset has only 40 trials per subject."
        )

    if rawdata.shape[1] <= max(SOURCE_EOG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: rawdata does not contain expected EOG channels.")

    X = np.asarray(X_eeg, dtype=np.float64)
    eog = np.asarray(rawdata[:, SOURCE_EOG_CHANNEL_INDICES, :], dtype=np.float64)

    n_trials, n_chans, n_samples = X.shape
    eog_2d = eog.transpose(0, 2, 1).reshape(-1, len(SOURCE_EOG_CHANNEL_INDICES))
    eeg_2d = X.transpose(0, 2, 1).reshape(-1, n_chans)

    design = np.column_stack([np.ones(eog_2d.shape[0]), eog_2d])
    beta, *_ = np.linalg.lstsq(design, eeg_2d, rcond=None)
    eog_contribution = design[:, 1:] @ beta[1:, :]
    corrected = eeg_2d - eog_contribution
    corrected = corrected.reshape(n_trials, n_samples, n_chans).transpose(0, 2, 1)

    steps.append("EOG correction: linear regression using HEOG/VEOG before dropping EOG")
    return corrected

def apply_source_artifact_clipping(X_eeg, subject_id, config, steps):
    mode = str(config.get("artifact_clip_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source artifact clipping skipped")
        return X, {"artifact_clip_applied": False, "artifact_clip_threshold": None}

    if mode == "absolute":
        threshold = config.get("artifact_clip_abs_value")
        if threshold is None:
            raise ValueError("artifact_clip_abs_value must be set when artifact_clip_mode='absolute'.")
        threshold = float(threshold)
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: absolute ±{threshold:g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    if mode == "percentile":
        pct = float(config.get("artifact_clip_percentile", 99.5))
        threshold = float(np.nanpercentile(np.abs(X), pct))
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: percentile {pct:g}% -> ±{threshold:.4g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_percentile": pct,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    raise ValueError(f"Unsupported artifact_clip_mode={config.get('artifact_clip_mode')}")

def apply_reference(raw, config, steps, timing_label):
    mode = str(config.get("reference_mode", "average")).lower()
    if mode in ("none", "off", "false"):
        steps.append(f"reference skipped at {timing_label}")
        return raw
    if mode == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
        steps.append(f"average reference at {timing_label}")
        return raw
    raise ValueError(f"Unsupported reference_mode={config.get('reference_mode')}")

def apply_notch(raw, config, steps, timing_label):
    freqs = config.get("notch_freqs", None)
    if freqs is None or freqs == []:
        return raw
    raw.notch_filter(freqs=freqs, verbose=False)
    steps.append(f"notch filter {freqs} Hz at {timing_label}")
    return raw

def apply_resample(raw, config, steps):
    if bool(config.get("resample", True)):
        target = float(config.get("resample_sfreq", 128))
        raw.resample(target, verbose=False)
        steps.append(f"resample to {target:g} Hz")
    else:
        steps.append("resample skipped; kept source 500 Hz")
    return raw

def _float_or_none_or_auto(value):
    if value is None:
        return None
    if isinstance(value, str) and value.lower() == "auto":
        return "auto"
    return float(value)

def apply_bandpass(raw, config, steps):
    if not bool(config.get("filter_enabled", True)):
        steps.append("bandpass skipped")
        return raw

    l_freq = config.get("filter_low", None)
    h_freq = config.get("filter_high", None)
    l_freq = None if l_freq is None else float(l_freq)
    h_freq = None if h_freq is None else float(h_freq)

    method = str(config.get("filter_method", "fir")).lower()
    if method == "iir":
        iir_params = config.get("filter_iir_params", None)
        if iir_params is None:
            iir_params = {"order": 2, "ftype": "butter"}
        raw.filter(l_freq=l_freq, h_freq=h_freq, method="iir", iir_params=iir_params, verbose=False)
        steps.append(f"IIR bandpass {l_freq}–{h_freq} Hz | params={iir_params}")
    elif method == "fir":
        raw.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            method="fir",
            phase=config.get("filter_phase", "zero"),
            fir_design=config.get("filter_fir_design", "firwin"),
            l_trans_bandwidth=_float_or_none_or_auto(config.get("filter_l_trans_bandwidth", "auto")),
            h_trans_bandwidth=_float_or_none_or_auto(config.get("filter_h_trans_bandwidth", "auto")),
            verbose=False,
        )
        steps.append(
            f"FIR bandpass {l_freq}–{h_freq} Hz | phase={config.get('filter_phase')} | "
            f"fir_design={config.get('filter_fir_design')}"
        )
    else:
        raise ValueError(f"Unsupported filter_method={config.get('filter_method')}")
    return raw

def apply_mne_raw_pipeline(raw, config, steps):
    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()

    if reference_timing == "before_resample_filter":
        raw = apply_reference(raw, config, steps, "before resample/filter")

    if bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "before resample/filter")

    raw = apply_resample(raw, config, steps)

    if reference_timing == "after_resample_before_filter":
        raw = apply_reference(raw, config, steps, "after resample before filter")

    raw = apply_bandpass(raw, config, steps)

    if reference_timing == "after_filter":
        raw = apply_reference(raw, config, steps, "after filter")

    if not bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "after bandpass")

    return raw

def maybe_reject_bad_trials(X_win, y, subject_id, config, steps):
    stats = {
        "reject_bad_trials": bool(config.get("reject_bad_trials", False)),
        "n_trials_before_reject": int(len(y)),
        "n_trials_after_reject": int(len(y)),
        "n_rejected_trials": 0,
        "rejection_skipped": False,
    }

    if not bool(config.get("reject_bad_trials", False)):
        steps.append("bad-trial rejection skipped")
        return X_win, y, stats

    keep = np.ones(len(y), dtype=bool)

    ptp_threshold = config.get("reject_peak_to_peak_threshold", None)
    if ptp_threshold is not None:
        ptp = np.ptp(X_win, axis=-1).max(axis=1)
        keep &= ptp <= float(ptp_threshold)
        stats["reject_peak_to_peak_threshold"] = float(ptp_threshold)
        stats["max_trial_peak_to_peak"] = float(np.max(ptp))

    abs_threshold = config.get("reject_abs_threshold", None)
    if abs_threshold is not None:
        max_abs = np.max(np.abs(X_win), axis=(1, 2))
        keep &= max_abs <= float(abs_threshold)
        stats["reject_abs_threshold"] = float(abs_threshold)
        stats["max_trial_abs"] = float(np.max(max_abs))

    proposed_y = y[keep]
    min_required = config.get("min_trials_per_class_after_reject", None)
    if min_required is None:
        if config.get("evaluation_mode") == "liu2024_repeated_60_40":
            min_required = 12
        else:
            min_required = int(config.get("cv_folds", 5))
    proposed_counts = np.bincount(proposed_y, minlength=TARGET_N_CLASSES)

    if len(proposed_y) == 0 or proposed_counts.min() < int(min_required):
        steps.append(
            "bad-trial rejection skipped because it would leave too few samples "
            f"per class: proposed_counts={proposed_counts.tolist()}, min_required={min_required}"
        )
        stats["rejection_skipped"] = True
        stats["proposed_class_counts_after_reject"] = proposed_counts.tolist()
        return X_win, y, stats

    X_new = X_win[keep]
    y_new = proposed_y
    stats["n_trials_after_reject"] = int(len(y_new))
    stats["n_rejected_trials"] = int(np.sum(~keep))
    stats["class_counts_after_reject"] = np.bincount(y_new, minlength=TARGET_N_CLASSES).tolist()
    steps.append(
        f"bad-trial rejection applied: rejected={stats['n_rejected_trials']} / {stats['n_trials_before_reject']} | "
        f"class_counts={stats['class_counts_after_reject']}"
    )
    return X_new, y_new, stats

def preprocess_subject_configurable(rawdata, labels, subject_id, config=None):
    """Apply the configured preprocessing pipeline to one Liu2024 source subject.

    Order:
      1. source-domain operations: channel selection, mean removal, detrending, EOG regression, clipping
      2. MNE RawArray operations: reference, notch, resample, bandpass
      3. window crop and optional trial rejection
      4. fold-safe normalization later inside training split code
    """
    config = CONFIG if config is None else config

    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    steps = describe_pipeline(build_preprocessing_pipeline(config))
    runtime_steps = []
    preprocessing_stats = {}

    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    runtime_steps.append("select 29 EEG channels; drop CPz source reference, EOG, and marker")

    X_eeg = apply_source_demean(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_source_detrend(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_eog_correction(X_eeg, rawdata, subject_id, config, runtime_steps)
    X_eeg, clip_stats = apply_source_artifact_clipping(X_eeg, subject_id, config, runtime_steps)
    preprocessing_stats.update(clip_stats)

    X_eeg_volts = source_values_to_mne_volts(X_eeg, config)
    runtime_steps.append(f"convert source {config.get('source_unit')} to MNE volts")

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    raw = apply_mne_raw_pipeline(raw, config, runtime_steps)

    effective_sfreq = float(raw.info["sfreq"])
    data = mne_volts_to_model_unit(raw.get_data(), config)
    runtime_steps.append(f"convert MNE volts to model {config.get('final_model_unit')}")

    expected_samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:  # type: ignore
        n_full = data.shape[1] // n_trials  # type: ignore
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]  # type: ignore
        runtime_steps.append(f"trim continuous samples to full trials: {expected_samples_per_trial} samples/trial")

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)  # type: ignore

    start_sample = int(round(float(config["mi_window_start_s"]) * effective_sfreq))
    window_samples = int(config["target_window_samples"]) if config.get("target_window_samples") is not None else int(round(float(config["target_window_s"]) * effective_sfreq))
    stop_sample = start_sample + window_samples

    if stop_sample > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop [{start_sample}:{stop_sample}] exceeds trial length "
            f"{X_rs.shape[-1]} at effective_sfreq={effective_sfreq}"
        )

    X_win = X_rs[:, :, start_sample:stop_sample]
    runtime_steps.append(f"crop fixed window samples [{start_sample}:{stop_sample}]")

    y = labels_to_zero_based(labels)
    X_win, y, reject_stats = maybe_reject_bad_trials(X_win, y, subject_id, config, runtime_steps)
    preprocessing_stats.update(reject_stats)

    # Save both the intended pipeline and the actual runtime steps.
    preprocessing_stats["pipeline_plan"] = steps
    preprocessing_stats["runtime_steps"] = runtime_steps

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial), runtime_steps, preprocessing_stats


[2026-06-04 19:09:21] Configured preprocessing pipeline:
[2026-06-04 19:09:21]   - [ON] source::select_eeg_channels | - select Liu EEG channels, drop CPz reference, EOG, and marker before model input
[2026-06-04 19:09:21]   - [off] source::demean | mode=none | baseline_window_s=[0.0, 2.0]
[2026-06-04 19:09:21]   - [off] source::detrend | mode=none
[2026-06-04 19:09:21]   - [off] source::eog_correction | mode=none
[2026-06-04 19:09:21]   - [off] source::artifact_clipping | mode=none | percentile=99.5
[2026-06-04 19:09:21]   - [ON] mne_raw::reference | timing=before_resample_filter | mode=average
[2026-06-04 19:09:21]   - [ON] mne_raw::resample | sfreq=128
[2026-06-04 19:09:21]   - [ON] mne_raw::bandpass_filter | l_freq=0.5 | h_freq=40.0 | method=fir | phase=zero | fir_design=firwin
[2026-06-04 19:09:21]   - [off] mne_raw::notch_filter | timing=after_bandpass
[2026-06-04 19:09:21]   - [ON] window::crop_fixed_mi_window | start_s=1.5 | target_window_samples=537
[2026-06-04 19:09:21]   - [o

In [10]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)
        if self.X.ndim != 3:
            raise ValueError(f"SubjectArrayDataset expects X as N x C x T, got shape={self.X.shape}.")
        if len(self.X) != len(self.y):
            raise ValueError(f"X/y length mismatch: {len(self.X)} windows vs {len(self.y)} labels.")

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected one EEG window as C x T, got shape={x.shape}.")
        return x, int(self.y[idx])


class FoldNormalizedDataset(Dataset):
    """Wrap a dataset and apply either fold-fitted or trial-wise normalization."""

    def __init__(self, dataset, normalizer_state):
        self.dataset = dataset
        self.normalizer_state = normalizer_state or {"mode": "none"}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = apply_normalizer_to_array(x, self.normalizer_state)
        if x.ndim != 2:
            raise ValueError(f"Normalization must return C x T, got shape={x.shape}.")
        return x.astype(np.float32), int(y)


class TrainAugmentedDataset(Dataset):
    """Lightweight training-time augmentation for tiny subject-level MI data.

    The augmentation is intentionally simple and shape-safe:
      - Gaussian noise proportional to per-window std
      - small random temporal roll
      - random channel dropout
    """

    def __init__(
        self,
        dataset,
        noise_fraction: float = 0.0,
        time_shift_samples: int = 0,
        channel_dropout_prob: float = 0.0,
    ):
        self.dataset = dataset
        self.noise_fraction = float(noise_fraction or 0.0)
        self.time_shift_samples = int(time_shift_samples or 0)
        self.channel_dropout_prob = float(channel_dropout_prob or 0.0)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = np.asarray(x, dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Augmentation expects C x T, got shape={x.shape}.")

        if self.time_shift_samples > 0:
            shift = np.random.randint(-self.time_shift_samples, self.time_shift_samples + 1)
            if shift != 0:
                x = np.roll(x, shift=shift, axis=-1)

        if self.channel_dropout_prob > 0:
            mask = np.random.rand(x.shape[0]) >= self.channel_dropout_prob
            if not mask.any():
                mask[np.random.randint(0, x.shape[0])] = True
            x = x.copy()
            x[~mask, :] = 0.0

        if self.noise_fraction > 0:
            sigma = self.noise_fraction * float(np.std(x) + 1e-8)
            noise = np.random.randn(*x.shape).astype(np.float32) * sigma
            x = x + noise

        return x.astype(np.float32), int(y)


# Backward compatibility for older code.
NoisyDataset = TrainAugmentedDataset


In [11]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
MAT_FILES = []

files = find_source_mat_files(SOURCE_EXTRACT_DIR)

if files:
    MAT_FILES = files

if not MAT_FILES:
    raise FileNotFoundError(
        "Could not find Liu2024 source .mat files. "
    )

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")

# Write a structure preview for the first subject. This makes it clear whether
# the local files expose top-level rawdata/labels or an `eeg` struct.
preview_path = ARTIFACT_DIR / "mat_structure_preview_first_subject.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"MAT structure preview saved to: {preview_path}")

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if CONFIG["subjects_to_use"] is not None and sid not in set(int(s) for s in CONFIG["subjects_to_use"]):
        continue
    if sid in set(int(s) for s in CONFIG["exclude_subjects"]):
        continue
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    validate_liu_source_subject(X_raw, y_raw, sid, path=p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects loaded.")

SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]
print(f"Subjects loaded: {SUBJECTS}")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
subjects_df.to_csv(subject_inventory_path, index=False)
print(f"Subject inventory saved to: {subject_inventory_path}")

EEG_INFO = make_liu_info(EFFECTIVE_SFREQ)
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []
preprocessing_steps_first_subject = None

for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, _, _ = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial, preprocessing_steps, preprocessing_stats = preprocess_subject_configurable(X_raw, y_raw, sid)

    if preprocessing_steps_first_subject is None:
        preprocessing_steps_first_subject = preprocessing_steps

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))
    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "preprocessed_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "target_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "source_unit": CONFIG["source_unit"],
        "final_model_unit": CONFIG["final_model_unit"],
        "demean_mode": CONFIG["demean_mode"],
        "reference_mode": CONFIG["reference_mode"],
        "reference_timing": CONFIG["reference_timing"],
        "resample": bool(CONFIG["resample"]),
        "effective_sfreq": float(EFFECTIVE_SFREQ),
        "filter_enabled": bool(CONFIG["filter_enabled"]),
        "filter_low": CONFIG.get("filter_low"),
        "filter_high": CONFIG.get("filter_high"),
        "filter_method": CONFIG.get("filter_method"),
        "mi_window_start_s": float(CONFIG["mi_window_start_s"]),
        "eog_correction": CONFIG.get("eog_correction"),
        "artifact_clip_mode": CONFIG.get("artifact_clip_mode"),
        "reject_bad_trials": bool(CONFIG.get("reject_bad_trials", False)),
        "normalization_mode": CONFIG.get("normalization_mode"),
        **preprocessing_stats,
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)
print(f"X_ALL shape: {X_ALL.shape} | Y_ALL counts: {np.bincount(Y_ALL).tolist()}")

if preprocessing_steps_first_subject is not None:
    print("Preprocessing steps used:")
    for step in preprocessing_steps_first_subject:
        print(f"  - {step}")

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_path = ARTIFACT_DIR / "window_counts_by_subject.csv"
window_summary_df.to_csv(window_summary_path, index=False)
print(f"Window summary saved to: {window_summary_path}")
display(window_summary_df.head())

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)


[2026-06-04 19:09:21] Source extract dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_data/liu2024_figshare/sourcedata
[2026-06-04 19:09:21] Found 50 .mat files
[2026-06-04 19:09:21] MAT structure preview saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/mat_structure_preview_first_subject.csv
[2026-06-04 19:09:28] Subjects loaded: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
[2026-06-04 19:09:28] Subject inventory saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/subject_inventory.csv
[2026-06-04 19:09:42] X_ALL shape: (2000, 29, 537) | Y_ALL coun

,subject_id,n_windows,class_counts,preprocessed_shape,resampled_samples_per_trial,crop_start_sample,crop_stop_sample,target_window_samples,effective_window_duration_s,source_unit,...,reject_bad_trials,normalization_mode,artifact_clip_applied,artifact_clip_threshold,n_trials_before_reject,n_trials_after_reject,n_rejected_trials,rejection_skipped,pipeline_plan,runtime_steps
0,1,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
1,2,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
2,3,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
3,4,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
4,5,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...


# 4. Model and Representation Extraction

In [12]:
def build_model():
    common_kwargs = {
        "n_chans": len(CH_NAMES),
        "chs_info": CHS_INFO,
        "n_times": WINDOW_SAMPLES,
        "n_outputs": TARGET_N_CLASSES,
    }
    mode = CONFIG["pretrained_mode"]
    if mode == "from_pretrained":
        model = SignalJEPA_PreLocal.from_pretrained(
            CONFIG["pretrained_repo_id"],
            **common_kwargs,
            strict=False,
        )
        info = {
            "loading_path": "from_pretrained",
            "repo_id": CONFIG["pretrained_repo_id"],
            "mode": mode,
        }
    elif mode == "random":
        model = SignalJEPA_PreLocal(**common_kwargs)
        info = {
            "loading_path": "random_initialization",
            "repo_id": None,
            "mode": mode,
        }
    else:
        raise ValueError("pretrained_mode must be 'from_pretrained' or 'random'.")
    info["model_name"] = CONFIG["model_name"]
    return model, info

def freeze_model(model):
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.eval()
    return model

def list_model_modules(model, max_rows=80):
    rows = []
    for name, module in model.named_modules():
        rows.append({"name": name, "type": type(module).__name__})
    df = pd.DataFrame(rows)
    module_path = ARTIFACT_DIR / "model_modules.csv"
    df.to_csv(module_path, index=False)
    print(f"Saved module list to: {module_path}")
    display(df.head(max_rows))
    return df

In [13]:
def _module_by_name(model, module_name):
    modules = dict(model.named_modules())
    if module_name in modules:
        return modules[module_name]
    matches = [name for name in modules if name.endswith(module_name)]
    if len(matches) == 1:
        return modules[matches[0]]
    if len(matches) > 1:
        raise ValueError(f"Multiple module matches for {module_name}: {matches}")
    raise ValueError(f"Could not find module {module_name}. See model_modules.csv.")

def _tensor_to_feature_matrix(tensor, pooling="flatten"):
    if isinstance(tensor, (tuple, list)):
        tensor = tensor[0]
    tensor = tensor.detach().cpu().float()
    if tensor.ndim == 1:
        tensor = tensor.unsqueeze(0)
    if tensor.ndim == 2:
        return tensor.numpy()
    if pooling == "mean_tokens" and tensor.ndim == 3:
        return tensor.mean(dim=1).numpy()
    return tensor.reshape(tensor.shape[0], -1).numpy()

def extract_representations(model, dataset, batch_size=None):
    batch_size = int(batch_size or CONFIG.get("representation_batch_size", 16))
    layer = str(CONFIG.get("representation_layer", "final_layer_input"))
    pooling = str(CONFIG.get("representation_pooling", "flatten"))
    model = freeze_model(model).to(DEVICE)

    captured = {}
    handles = []

    if layer == "final_layer_input":
        final_layer = _module_by_name(model, "final_layer")
        def pre_hook(module, inputs):
            captured["features"] = inputs[0]
        handles.append(final_layer.register_forward_pre_hook(pre_hook))
        use_model_output = False
    elif layer == "model_output":
        use_model_output = True
    else:
        module = _module_by_name(model, layer)
        def hook(module, inputs, output):
            captured["features"] = output
        handles.append(module.register_forward_hook(hook))
        use_model_output = False

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    features = []
    with torch.no_grad():
        for batch in loader:
            X = batch[0].float().to(DEVICE)
            captured.clear()
            output = model(X)
            tensor = output if use_model_output else captured.get("features", None)
            if tensor is None:
                raise RuntimeError(f"No features captured for representation_layer={layer}.")
            features.append(_tensor_to_feature_matrix(tensor, pooling=pooling))
    for handle in handles:
        handle.remove()
    return np.concatenate(features, axis=0)

def build_simple_classifier(name):
    name = str(name).lower()
    if name == "logistic_regression":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000, class_weight="balanced", solver="liblinear"),
        )
    if name == "linear_svm":
        return make_pipeline(
            StandardScaler(),
            LinearSVC(class_weight="balanced", max_iter=10000),
        )
    if name == "ridge":
        return make_pipeline(
            StandardScaler(),
            RidgeClassifier(class_weight="balanced"),
        )
    if name == "lda":
        return make_pipeline(
            StandardScaler(),
            LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
        )
    raise ValueError(f"Unsupported simple classifier: {name}")

# 5. Linear Probe Evaluation

In [14]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def dataset_to_array(dataset):
    X = np.stack([np.asarray(dataset[i][0], dtype=np.float32) for i in range(len(dataset))], axis=0)
    if X.ndim != 3:
        raise ValueError(f"Expected dataset windows to stack as N x C x T, got shape={X.shape}.")
    return X

def _safe_scale(scale, eps):
    scale = np.asarray(scale, dtype=np.float32)
    return np.where(np.abs(scale) < float(eps), 1.0, scale).astype(np.float32)

def _as_window_2d(x):
    """Keep each EEG example in C x T format for Braindecode SignalJEPA_PreLocal."""
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 3 and x.shape[0] == 1:
        x = x[0]
    if x.ndim != 2:
        raise ValueError(f"Expected one EEG window to have shape C x T, got shape={x.shape}.")
    return x

def _state_array_for_window(value, x):
    """Convert fitted fold statistics to shapes that broadcast over a single C x T window."""
    arr = np.asarray(value, dtype=np.float32)
    if arr.ndim == 0:
        return arr
    if x.ndim == 2 and arr.ndim == 3 and arr.shape[0] == 1:
        # Old shape from fold fitting was 1 x C x 1. For one example, use C x 1.
        arr = arr[0]
    if x.ndim == 2 and arr.ndim == 1 and arr.shape[0] == x.shape[0]:
        arr = arr[:, None]
    return arr

def apply_normalizer_to_array(x, state):
    """Apply a fitted fold normalizer or trial-wise normalizer to one C x T example.

    Important: this function must return C x T, not 1 x C x T.
    Returning 1 x C x T makes the DataLoader batch 4D and breaks SignalJEPA_PreLocal.
    """
    mode = str((state or {}).get("mode", "none")).lower()
    eps = float((state or {}).get("eps", CONFIG.get("normalization_eps", 1e-6)))
    x = _as_window_2d(x)

    if mode in ("none", "off", "false"):
        return x

    if mode == "train_global_zscore":
        mean = _state_array_for_window(state["mean"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - mean) / scale)

    if mode == "train_channel_zscore":
        mean = _state_array_for_window(state["mean"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - mean) / scale)

    if mode == "train_channel_robust":
        median = _state_array_for_window(state["median"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - median) / scale)

    if mode == "trial_global_zscore":
        mean = x.mean(keepdims=True)
        scale = max(float(x.std()), eps)
        return _as_window_2d((x - mean) / scale)

    if mode == "trial_channel_zscore":
        mean = x.mean(axis=-1, keepdims=True)
        scale = _safe_scale(x.std(axis=-1, keepdims=True), eps)
        return _as_window_2d((x - mean) / scale)

    raise ValueError(f"Unsupported normalization_mode={mode}")

def fit_fold_normalizer(train_dataset):
    """Fit normalization on the training fold only where applicable."""
    mode = str(CONFIG.get("normalization_mode", "none")).lower()
    eps = float(CONFIG.get("normalization_eps", 1e-6))
    state = {"mode": mode, "eps": eps}
    summary = {"mode": mode, "fit_scope": "none"}

    if mode in ("none", "off", "false", "trial_global_zscore", "trial_channel_zscore"):
        if mode.startswith("trial_"):
            summary["fit_scope"] = "trial_only_no_fold_fit"
        return state, summary

    X = dataset_to_array(train_dataset)  # N x C x T

    if mode == "train_global_zscore":
        mean = np.asarray(X.mean(), dtype=np.float32)
        scale = np.asarray(max(float(X.std()), eps), dtype=np.float32)
        state.update({"mean": mean, "scale": scale})
        summary.update({
            "fit_scope": "training_fold",
            "mean_shape": [],
            "scale_shape": [],
            "train_mean": float(mean),
            "train_scale": float(scale),
        })
        return state, summary

    if mode == "train_channel_zscore":
        # Shape is C x 1 so it broadcasts correctly over one C x T window.
        mean = X.mean(axis=(0, 2)).astype(np.float32)[:, None]
        scale = _safe_scale(X.std(axis=(0, 2)).astype(np.float32)[:, None], eps)
        state.update({"mean": mean, "scale": scale})
        summary.update({
            "fit_scope": "training_fold",
            "mean_shape": list(mean.shape),
            "scale_shape": list(scale.shape),
            "mean_scale_min": float(scale.min()),
            "mean_scale_max": float(scale.max()),
        })
        return state, summary

    if mode == "train_channel_robust":
        # Shape is C x 1 so it broadcasts correctly over one C x T window.
        median = np.median(X, axis=(0, 2)).astype(np.float32)[:, None]
        q75 = np.percentile(X, 75, axis=(0, 2)).astype(np.float32)[:, None]
        q25 = np.percentile(X, 25, axis=(0, 2)).astype(np.float32)[:, None]
        iqr = _safe_scale((q75 - q25).astype(np.float32), eps)
        state.update({"median": median, "scale": iqr})
        summary.update({
            "fit_scope": "training_fold",
            "median_shape": list(median.shape),
            "scale_shape": list(iqr.shape),
            "iqr_min": float(iqr.min()),
            "iqr_max": float(iqr.max()),
        })
        return state, summary

    raise ValueError(f"Unsupported normalization_mode={CONFIG.get('normalization_mode')}")

def maybe_wrap_normalized(train_set, test_set):
    state, summary = fit_fold_normalizer(train_set)
    mode = str(summary.get("mode", "none")).lower()
    if mode in ("none", "off", "false"):
        return train_set, test_set, summary
    return FoldNormalizedDataset(train_set, state), FoldNormalizedDataset(test_set, state), summary



def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

def compute_collapse_diagnostics(y_pred, n_classes):
    pred_hist = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    n_pred = int(pred_hist.sum())
    collapse_ratio = float(pred_hist.max() / n_pred) if n_pred else 0.0
    threshold = float(CONFIG.get("collapse_threshold", 0.90))
    return {
        "prediction_histogram": pred_hist.tolist(),
        "collapse_ratio": float(collapse_ratio),
        "collapse_threshold": threshold,
        "collapse_flag": bool(collapse_ratio >= threshold),
        "majority_predicted_class": int(pred_hist.argmax()) if n_pred else None,
    }

In [15]:
def run_representation_probe(train_set, test_set, fold_id, fold_label, n_total_folds=None):
    if CONFIG["set_seed"]:
        seed_everything(BASE_SEED)

    y_train = get_targets(train_set)
    y_test = get_targets(test_set)
    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    train_set, test_set, normalization_summary = maybe_wrap_normalized(train_set, test_set)

    model, model_info = build_model()
    model = freeze_model(model)

    fold_tag = f"/{n_total_folds}" if n_total_folds is not None else ""
    print(f"\nFold {fold_id}{fold_tag} | {fold_label}")
    print(f"    Train: {len(train_set)} | counts={train_counts.tolist()}")
    print(f"    Test:  {len(test_set)} | counts={test_counts.tolist()}")
    print(f"    Representation layer: {CONFIG.get('representation_layer')} | pooling={CONFIG.get('representation_pooling')}")
    print(f"    Normalization: {normalization_summary.get('mode')} | fit_scope={normalization_summary.get('fit_scope')}")

    X_train_rep = extract_representations(model, train_set)
    X_test_rep = extract_representations(model, test_set)

    results = []
    for classifier_name in CONFIG.get("simple_classifiers", ["logistic_regression"]):
        try:
            clf = build_simple_classifier(classifier_name)
            clf.fit(X_train_rep, y_train)
            y_pred = clf.predict(X_test_rep)

            metrics = compute_classification_metrics(y_test, y_pred)
            cm = confusion_matrix(y_test, y_pred, labels=list(range(TARGET_N_CLASSES))).tolist()
            collapse = compute_collapse_diagnostics(y_pred, TARGET_N_CLASSES)
            error = None
        except Exception as exc:
            print(f"    {classifier_name:20s} failed: {exc}")
            metrics = {"accuracy": np.nan, "balanced_accuracy": np.nan}
            cm = None
            collapse = {
                "prediction_histogram": None,
                "collapse_ratio": np.nan,
                "collapse_threshold": float(CONFIG.get("collapse_threshold", 0.90)),
                "collapse_flag": False,
                "majority_predicted_class": None,
            }
            error = str(exc)

        result = {
            "subject_id": str(getattr(train_set, "subject_id", "unknown")),
            "fold_id": int(fold_id),
            "fold_label": str(fold_label),
            "classifier_name": str(classifier_name),
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "confusion_matrix": cm,
            "collapse": collapse,
            "error": error,
            "train_counts": train_counts.tolist(),
            "test_counts": test_counts.tolist(),
            "representation": {
                "layer": CONFIG.get("representation_layer"),
                "pooling": CONFIG.get("representation_pooling"),
                "train_shape": list(X_train_rep.shape),
                "test_shape": list(X_test_rep.shape),
            },
            "normalization": normalization_summary,
            "model_info": model_info,
        }
        if error is None:
            print(
                f"    {classifier_name:20s} "
                f"acc={metrics['accuracy']:.4f} bal_acc={metrics['balanced_accuracy']:.4f} "
                f"collapse={collapse['collapse_ratio']:.3f}"
            )
        results.append(result)
    return results

def run_subject_cv(subject_id, dataset, n_classes):
    y = get_targets(dataset)
    if len(np.unique(y)) < n_classes:
        print(f"WARNING: subject {subject_id} has fewer than {n_classes} classes. Skipping.")
        return []

    cv = StratifiedKFold(
        n_splits=int(CONFIG["cv_folds"]),
        shuffle=True,
        random_state=int(CONFIG["cv_random_state"]),
    )

    subject_results = []
    splits = list(cv.split(np.zeros(len(y)), y))
    for fold_idx, (train_idx, test_idx) in enumerate(splits, start=1):
        train_set = Subset(dataset, train_idx.tolist())
        test_set = Subset(dataset, test_idx.tolist())
        train_set.subject_id = str(subject_id)
        test_set.subject_id = str(subject_id)
        subject_results.extend(
            run_representation_probe(
                train_set,
                test_set,
                fold_id=fold_idx,
                fold_label=f"subject={subject_id}",
                n_total_folds=len(splits),
            )
        )
    return subject_results

In [16]:
print("=" * 70)
print("STARTING S-JEPA REPRESENTATION LINEAR PROBE")
print("=" * 70)
print(f"Experiment:    {CONFIG.get('experiment_name')}")
print(f"Subjects:      {sorted(SUBJECT_WINDOWS.keys(), key=_sort_subject_key)}")
print(f"Model:         {CONFIG['model_name']}")
print(f"Pretrained:    {CONFIG['pretrained_mode']}")
print(f"Representation:{CONFIG.get('representation_layer')} | pooling={CONFIG.get('representation_pooling')}")
print(
    "Preprocessing: "
    f"demean={CONFIG['demean_mode']} | ref={CONFIG['reference_mode']}@{CONFIG.get('reference_timing')} | "
    f"resample={CONFIG['resample']}->{CONFIG.get('resample_sfreq')} | "
    f"filter={CONFIG['filter_enabled']} {CONFIG.get('filter_low')}–{CONFIG.get('filter_high')} {CONFIG.get('filter_method')}"
)
print(f"Evaluation:    {CONFIG.get('evaluation_mode')} | cv_folds={CONFIG.get('cv_folds')}")
print(f"Window:        start={CONFIG['mi_window_start_s']}s | samples={WINDOW_SAMPLES} | duration={TARGET_TRIAL_DURATION_S:.4f}s | effective sfreq={EFFECTIVE_SFREQ}")
print(f"Device:        {DEVICE}")
print("=" * 70)

_debug_model, _ = build_model()
MODULE_TABLE = list_model_modules(_debug_model)
del _debug_model

FOLD_RESULTS = []
for sid in sorted(SUBJECT_WINDOWS.keys(), key=_sort_subject_key):
    FOLD_RESULTS.extend(run_subject_cv(sid, SUBJECT_WINDOWS[sid], TARGET_N_CLASSES))
print(f"\nTotal classifier-fold results completed: {len(FOLD_RESULTS)}")

[2026-06-04 19:09:42] ======================================================================
[2026-06-04 19:09:42] STARTING S-JEPA REPRESENTATION LINEAR PROBE
[2026-06-04 19:09:42] ======================================================================
[2026-06-04 19:09:42] Experiment:    liu2024_sjepa_representation_linear_probe
[2026-06-04 19:09:42] Subjects:      ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50']
[2026-06-04 19:09:42] Model:         SignalJEPA_PreLocal
[2026-06-04 19:09:42] Pretrained:    from_pretrained
[2026-06-04 19:09:42] Representation:final_layer_input | pooling=flatten
[2026-06-04 19:09:42] Preprocessing: demean=none | ref=average@before_resample_filter | resample=True->128 | filter=True 0.5–40.0 fir
[2026-06-04 19:09:42] E

[2026-06-04 19:09:43] Saved module list to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/model_modules.csv


,name,type
0,,SignalJEPA_PreLocal
1,feature_encoder,_ConvFeatureEncoder
2,feature_encoder.0,Rearrange
3,feature_encoder.1,Sequential
4,feature_encoder.1.0,Conv1d
5,feature_encoder.1.1,Dropout
6,feature_encoder.1.2,GroupNorm
7,feature_encoder.1.3,GELU
8,feature_encoder.2,Sequential
9,feature_encoder.2.0,Conv1d



[2026-06-04 19:09:44] Fold 1/5 | subject=1
[2026-06-04 19:09:44]     Train: 32 | counts=[16, 16]
[2026-06-04 19:09:44]     Test:  8 | counts=[4, 4]
[2026-06-04 19:09:44]     Representation layer: final_layer_input | pooling=flatten
[2026-06-04 19:09:44]     Normalization: none | fit_scope=none
[2026-06-04 19:09:45]     logistic_regression  acc=0.6250 bal_acc=0.6250 collapse=0.625
[2026-06-04 19:09:45]     linear_svm           acc=0.6250 bal_acc=0.6250 collapse=0.625
[2026-06-04 19:09:45]     ridge                acc=0.6250 bal_acc=0.6250 collapse=0.625
[2026-06-04 19:09:45]     lda                  acc=0.6250 bal_acc=0.6250 collapse=0.625

[2026-06-04 19:09:45] Fold 2/5 | subject=1
[2026-06-04 19:09:45]     Train: 32 | counts=[16, 16]
[2026-06-04 19:09:45]     Test:  8 | counts=[4, 4]
[2026-06-04 19:09:45]     Representation layer: final_layer_input | pooling=flatten
[2026-06-04 19:09:45]     Normalization: none | fit_scope=none
[2026-06-04 19:09:45]     logistic_regression  acc=0.500

# 6. Results

In [17]:
def aggregate_results(fold_results):
    if not fold_results:
        return {}, {}

    rows = []
    for result in fold_results:
        rows.append({
            "subject_id": result["subject_id"],
            "fold_id": result["fold_id"],
            "classifier_name": result["classifier_name"],
            "accuracy": result["accuracy"],
            "balanced_accuracy": result["balanced_accuracy"],
            "collapse_ratio": result["collapse"]["collapse_ratio"],
            "collapse_flag": result["collapse"]["collapse_flag"],
            "representation_layer": result["representation"]["layer"],
            "representation_pooling": result["representation"]["pooling"],
            "feature_dim": result["representation"]["train_shape"][1],
        })
    fold_df = pd.DataFrame(rows)
    subject_df = (
        fold_df
        .groupby(["classifier_name", "subject_id"], as_index=False)
        .agg(
            mean_accuracy=("accuracy", "mean"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            collapse_rate=("collapse_flag", "mean"),
            mean_collapse_ratio=("collapse_ratio", "mean"),
            n_folds=("fold_id", "count"),
        )
    )
    global_df = (
        fold_df
        .groupby("classifier_name", as_index=False)
        .agg(
            mean_accuracy=("accuracy", "mean"),
            std_accuracy=("accuracy", "std"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            std_balanced_accuracy=("balanced_accuracy", "std"),
            collapse_rate=("collapse_flag", "mean"),
            mean_collapse_ratio=("collapse_ratio", "mean"),
            n_results=("fold_id", "count"),
        )
        .sort_values("mean_balanced_accuracy", ascending=False)
    )

    display(global_df)
    display(subject_df.head())
    return fold_df, subject_df, global_df

FOLD_LEVEL_DF, SUBJECT_LEVEL_DF, GLOBAL_DF = aggregate_results(FOLD_RESULTS)

,classifier_name,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,collapse_rate,mean_collapse_ratio,n_results
0,lda,0.5585,0.173597,0.5585,0.173597,0.016,0.6455,250
3,ridge,0.5575,0.172118,0.5575,0.172118,0.012,0.6465,250
1,linear_svm,0.5540,0.169777,0.5540,0.169777,0.012,0.6470,250
2,logistic_regression,0.5515,0.176166,0.5515,0.176166,0.020,0.6495,250


,classifier_name,subject_id,mean_accuracy,mean_balanced_accuracy,collapse_rate,mean_collapse_ratio,n_folds
0,lda,1,0.650,0.650,0.0,0.600,5
1,lda,10,0.525,0.525,0.0,0.575,5
2,lda,11,0.450,0.450,0.0,0.650,5
3,lda,12,0.550,0.550,0.2,0.750,5
4,lda,13,0.775,0.775,0.0,0.625,5


# 7. Save Artifacts

In [18]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(FOLD_RESULTS, f, indent=2)

fold_path = ARTIFACT_DIR / "fold_level_results.csv"
subject_path = ARTIFACT_DIR / "subject_level_summary.csv"
global_path = ARTIFACT_DIR / "global_method_comparison.csv"

FOLD_LEVEL_DF.to_csv(fold_path, index=False)
SUBJECT_LEVEL_DF.to_csv(subject_path, index=False)
GLOBAL_DF.to_csv(global_path, index=False)

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "config": CONFIG,
    "n_subjects": len(SUBJECT_WINDOWS),
    "n_results": len(FOLD_RESULTS),
    "artifacts": {
        "cv_results": str(cv_results_path),
        "fold_level_results": str(fold_path),
        "subject_level_summary": str(subject_path),
        "global_method_comparison": str(global_path),
        "model_modules": str(ARTIFACT_DIR / "model_modules.csv"),
    },
}
metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)

print("Saved artifacts:")
for key, value in run_metadata["artifacts"].items():
    print(f"  {key}: {value}")

[2026-06-04 19:13:16] Saved artifacts:
[2026-06-04 19:13:16]   cv_results: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/cv_results.json
[2026-06-04 19:13:16]   fold_level_results: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/fold_level_results.csv
[2026-06-04 19:13:16]   subject_level_summary: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/subject_level_summary.csv
[2026-06-04 19:13:16]   global_method_comparison: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-representation-linear-probe/20260604_1909_76736b0d/global_method_comparison.csv
[2026-06-04 19:13:16] 